In [ ]:
import os
import torch
from decord import VideoReader, cpu
import matplotlib.pyplot as plt
from holmesvau.holmesvau_utils import load_model, generate, caption_clip, show_smapled_video
from holmesvau.clip_selection import select_clips, snippets_to_frames, load_gt_segments

mllm_path = 'ppxin321/HolmesVAU-2B'
sampler_path = '/home/emogenai4e/emo/VidAnomalyRetrieval/DescriptionModule/Holmes-VAU-ATS/anomaly_scorer.pth'
device = torch.device('cuda')
model, tokenizer, generation_config, sampler = load_model(mllm_path, sampler_path, device)

In [ ]:
video_path = "/home/emogenai4e/emo/VidAnomalyRetrieval/UCF_Video/Abuse/Abuse028_x264.mp4"
gt_annotation_file = "/home/emogenai4e/emo/VidAnomalyRetrieval/DescriptionModule/VadCLIP/list/Temporal_Anomaly_Annotation.txt"

K = 3
clip_sec = 16.0
snippet_size = 16  # dense_sample_freq inside generate()
select_frames = 12

clip_prompt = """Describe this surveillance video for a search database.

Extract concrete visual facts. Output ONE sentence (15-25 words) covering as many of these as visible:
- Time of day (at night / in the evening / during the day) — only if clearly visible
- Main subject(s): for people use clothing color + garment (shirt/hoodie/jacket/hat) + gender if clear; for vehicles use color + type (car/van/truck/motorcycle); for objects use type + state
- ONE main action verb (concrete: broke, kicked, robbed, collided, set fire, smashed, stole, beat, drove, entered, lit, shot)
- Direct object or target if any (door, window, gun, ATM, person, bag)
- Location with preposition (in the store / on the side of the road / in the parking lot / inside a room / at the gate)

STRICT RULES:
- Describe only what is literally visible. Never infer intent, cause, or outcome.
- Forbidden words: suspicious, suggests, appears, potential, seems, anomaly, event, activity, behavior, indicating, the video shows, depicts.
- No preamble. No \"this video shows\". Start directly with the description.
- If you are uncertain about a detail (color, gender, count), omit it rather than guess.

Output only the sentence."""

video_prompt = clip_prompt  # same template; swap if you want a wider summary

In [ ]:
# 1) Whole-video caption + dense anomaly score (via ATS dense pre-pass)
video_pred, _, video_frame_indices, anomaly_score = generate(
    video_path, video_prompt, model, tokenizer, generation_config, sampler,
    dense_sample_freq=snippet_size, select_frames=select_frames, use_ATS=True,
)
print('\n[VIDEO]', video_pred)

In [ ]:
# 2) Cut clips from the anomaly score and caption each one
vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
fps = float(vr.get_avg_fps())
max_frame = len(vr)

clips_snip = select_clips(anomaly_score, K=K, clip_sec=clip_sec, fps=fps, snippet_size=snippet_size)
clips_frame = snippets_to_frames(clips_snip, snippet_size=snippet_size, max_frame=max_frame)
print('Clips (snippet):', clips_snip)
print('Clips (sec):    ', [(round(s * snippet_size / fps, 2), round(e * snippet_size / fps, 2)) for s, e in clips_snip])

clip_results = []
for i, frame_range in enumerate(clips_frame):
    pred, frame_idx = caption_clip(vr, frame_range, clip_prompt, model, tokenizer, generation_config,
                                   select_frames=select_frames)
    clip_results.append({'snippet_range': clips_snip[i], 'frame_range': frame_range,
                         'frame_indices': frame_idx, 'caption': pred})
    print(f'\n[CLIP {i}] frames={frame_range}  ({frame_range[0]/fps:.2f}s - {frame_range[1]/fps:.2f}s)')
    print(' ', pred)

In [ ]:
# 3) Visualize: anomaly score, GT (red), selected clips (orange), sampled frames (green)
gt_segments = load_gt_segments(video_path, gt_annotation_file)
print('GT segments (frame):', gt_segments)

show_smapled_video(vr, video_frame_indices)
for i, r in enumerate(clip_results):
    print(f'\n[CLIP {i}] sampled frames:', r['frame_indices'])
    show_smapled_video(vr, r['frame_indices'])

if anomaly_score is not None:
    plt.figure(figsize=(8, 2))
    plt.plot(anomaly_score)
    for j, (gs, ge) in enumerate(gt_segments):
        plt.axvspan(gs / snippet_size, ge / snippet_size, color='red', alpha=0.25,
                    label='ground truth' if j == 0 else None)
    for j, (cs, ce) in enumerate(clips_snip):
        plt.axvspan(cs, ce, color='orange', alpha=0.2,
                    label='selected clip' if j == 0 else None)
    for idx in video_frame_indices:
        plt.vlines(idx / snippet_size, 0, 1, colors='green', linewidth=0.8)
    plt.ylim(0, 1)
    plt.xlabel('snippet')
    plt.ylabel('anomaly score')
    plt.legend(loc='upper right', fontsize=8)
    plt.show()